# Choosing a model locally: cost, latency and limits with Ollama

By the end you can:

- run two local models on the same ten emails and record what each one did;
- say what "cost" means when nothing is billed;
- explain the limits of local inference and where they bite;
- pick one model and defend the choice with numbers.

## Setup

**1. Ollama**, from ollama.com. On Linux:

```bash
curl -fsSL https://ollama.com/install.sh | sh
```

**2. Both models** (~10 GB together):

```bash
ollama pull qwen2.5:7b
ollama pull llama3.1:8b
ollama list
```

**3. One model in memory at a time** — two 7B models do not both fit in 16 GB next to Docker and a browser:

```bash
export OLLAMA_MAX_LOADED_MODELS=1
export OLLAMA_CONTEXT_LENGTH=8192
export OLLAMA_KEEP_ALIVE=5m
```

Those only work for a server started from **this** shell. Ollama normally runs as a service, which reads none of them — set them where it does look:

```bash
sudo systemctl edit ollama     # [Service] Environment="OLLAMA_MAX_LOADED_MODELS=1"
sudo systemctl restart ollama
```

Skip this and the cap is silently 3 models, which on a 16 GB laptop is how the second model fails with a `500`.

**4. Python side:** `uv pip install httpx2`

Ollama serves on `http://127.0.0.1:11434`. No key, no account, no card.

## The task, with the answers written first

Ten inbox emails, one question each: which queue does this belong to?

Write the expected answer **before** running anything. A label invented after seeing the output is not a test. Two of the ten are in Pidgin on purpose — local models are weaker there, and that belongs in the numbers.

What must stay fixed, or the comparison means nothing:

| Fixed | Why |
|---|---|
| the 10 emails | otherwise you compared prompts, not models |
| the system prompt | one prompt, one task |
| `temperature: 0` | no creativity wanted in a classifier |
| `num_ctx: 8192` | context changes both memory and speed |

In [1]:
EMAILS = [
    {"id": "E01", "expect": "billing", "subject": "Payment failed twice",
     "body": "My card was declined twice this morning. I need the invoice paid today."},
    {"id": "E02", "expect": "access", "subject": "Cannot log in after password reset",
     "body": "I reset my password and now the login page says my account is locked."},
    {"id": "E03", "expect": "general", "subject": "Question about the monthly plan",
     "body": "Does the basic plan include two users or only one?"},
    {"id": "E04", "expect": "urgent", "subject": "URGENT: production down",
     "body": "The API has returned 500 for forty minutes. Nobody can check out."},
    {"id": "E05", "expect": "billing", "subject": "Invoice 4402 is wrong",
     "body": "We were charged for eleven seats but only use seven."},
    {"id": "E06", "expect": "spam", "subject": "Congratulations you won",
     "body": "Click this link to claim your prize of five hundred thousand dollars."},
    {"id": "E07", "expect": "general", "subject": "Adding a teammate",
     "body": "How do I invite someone from my team to the workspace?"},
    {"id": "E08", "expect": "billing", "subject": "Dem don charge me two times",
     "body": "Abeg my card no dey work well, e don charge me two times this week."},
    {"id": "E09", "expect": "access", "subject": "Reporting tab no dey open",
     "body": "I be admin but the reports tab still dey tell me say I no get permission."},
    {"id": "E10", "expect": "general", "subject": "Meeting notes from Tuesday",
     "body": "Sharing the notes from our call in case anyone missed the decisions."},
]

print(len(EMAILS), "emails,", sum(1 for e in EMAILS if e["expect"] == "billing"), "of them billing")

10 emails, 3 of them billing


In [2]:
MODELS = {
    "qwen2.5:7b": {"params": "7B", "download_gb": 4.7, "ram_gb": 5.7, "licence": "Apache 2.0"},
    "llama3.1:8b": {"params": "8B", "download_gb": 4.9, "ram_gb": 6.0,
                    "licence": "Meta Llama 3.1 Community"},
}

# what these tokens would cost on a hosted 8B-class API, USD per 1M tokens.
# EDIT and DATE this if you price against a real provider.
HOSTED_REFERENCE = {"price_in": 0.05, "price_out": 0.08}

for name, info in MODELS.items():
    print(f"{name:<14} {info['params']:<4} {info['download_gb']} GB download, "
          f"{info['ram_gb']} GB in memory, {info['licence']}")

qwen2.5:7b     7B   4.7 GB download, 5.7 GB in memory, Apache 2.0
llama3.1:8b    8B   4.9 GB download, 6.0 GB in memory, Meta Llama 3.1 Community


Read the memory column against your laptop, not against the other model. `5.7 + 6.0 = 11.7 GB` — fine for one model on 16 GB, impossible for two, which is what `OLLAMA_MAX_LOADED_MODELS=1` enforces.

And a 7B model does not weigh 5.7 GB as text. It ships **quantized**: parameters stored in 4 bits instead of 16. That is why it fits on a laptop, and it is a trade — less memory and more speed, slightly less accuracy.

```bash
ollama show qwen2.5:7b        # parameters, quantization, context
```

In [3]:
import httpx2

r = httpx2.post("http://127.0.0.1:11434/api/show", json={"model": "qwen2.5:7b"}, timeout=30)
details = r.json()["details"]

for key in ("family", "parameter_size", "quantization_level"):
    print(f"{key:<20} {details.get(key)}")

family               qwen2
parameter_size       7.6B
quantization_level   Q4_K_M


`q4_K_M` is the default tag: 4-bit, medium quality. That is why the download is 4.7 GB and not 15 GB.

So a quantized 7B is **not** the model in the published benchmarks — yours are the numbers that count. Higher quantizations (`q8_0`) exist and cost memory and speed; if accuracy is short that is a lever, but it is a different comparison, so don't mix it into this one.

## The four numbers Ollama gives back

| Field | What it is | Unit |
|---|---|---|
| `prompt_eval_count` | your system prompt + the email | tokens |
| `eval_count` | the answer | tokens |
| `total_duration` | the whole call | **nanoseconds** |
| `message.content` | the answer | text |

Nanoseconds is why the wrapper divides by `1e9`. Miss that and every latency is a billion times too large — a mistake that looks like a broken laptop.

You never count tokens by hand; Ollama reports both.

**One model at a time.** Ollama keeps a model loaded for `OLLAMA_KEEP_ALIVE` after its last call, so without a cap it holds both. The trap that follows: a model that is *not* loaded pays a load time on its first call — which is the difference between run 1 and run 2 below.

## `ask()` — one function, one email

Everything the two scripts do is built on this. Read it once, properly:

- `build_prompt` puts subject and body into one string — the only thing that changes per email.
- `options` carries `temperature: 0` and `num_ctx: 8192`, the two settings held constant.
- `timeout=300` because CPU inference can take minutes; the 5-second default would fail every call.
- `perf_counter()` starts immediately before the request and stops immediately after — not around the loop, not around the parsing.
- the return dict is **the record**: answer, token counts, seconds. Everything downstream reads this shape.

In [4]:
import time

OLLAMA_CHAT = "http://127.0.0.1:11434/api/chat"
OLLAMA_GENERATE = "http://127.0.0.1:11434/api/generate"
OLLAMA_V1 = "http://127.0.0.1:11434/v1/chat/completions"

SYSTEM = ("You label inbox emails. Reply with exactly one word, one of: "
          "urgent, billing, access, general, spam.")


def build_prompt(email):
    return "Subject: " + email["subject"] + "\n\n" + email["body"]


def ask(model, email):
    prompt = build_prompt(email)

    start = time.perf_counter()
    response = httpx2.post(
        OLLAMA_CHAT,
        json={
            "model": model,
            "messages": [
                {"role": "system", "content": SYSTEM},
                {"role": "user", "content": prompt},
            ],
            "stream": False,
            "options": {"temperature": 0, "num_ctx": 8192},
        },
        timeout=300,
    )
    seconds = time.perf_counter() - start

    if response.status_code != 200:
        print("server said:", response.text[:300])
    response.raise_for_status()
    data = response.json()

    return {
        "id": email["id"],
        "prompt": prompt,
        "expect": email["expect"],
        "answer": data["message"]["content"].strip(),
        "tokens_in": data.get("prompt_eval_count", 0),
        "tokens_out": data.get("eval_count", 0),
        "seconds": round(seconds, 3),
    }

In [5]:
def unload(model):
    """Drop the model from memory now, instead of waiting for keep_alive to expire."""
    httpx2.post(OLLAMA_GENERATE, json={"model": model, "keep_alive": 0}, timeout=60)


try:
    installed = [m["name"] for m in httpx2.get(
        "http://127.0.0.1:11434/api/tags", timeout=5).json()["models"]]
    print("server up --", len(installed), "models")
    for name in MODELS:
        print(" ", name, "ready" if name in installed else "MISSING -- ollama pull " + name)
except httpx2.ConnectError:
    print("server not running -> open the Ollama app, or run: ollama serve")

server up -- 2 models
  qwen2.5:7b ready
  llama3.1:8b ready


`unload()` posts to `/api/generate` with `keep_alive: 0` — the API version of `ollama stop <model>`. The check asks the **server** what it can serve right now; `ollama list` only shows what is on disk, which is a different question.

## Two runs, and what gets written

- **run 1** — the first call loads the model, the rest are normal. Slower, and it contains the load.
- **run 2** — the model is already in memory. This is what one more email actually costs.

Quote run 2 as your latency, keep run 1 as evidence of the load penalty, then unload so the next model starts clean.

Each model writes `results/<model>.json` — the same file the standalone scripts produced, so notebook and scripts agree.

In [6]:
import json
from pathlib import Path

RESULTS = Path("results")
RESULTS.mkdir(exist_ok=True)


def run_pass(model, run_label):
    rows = []
    for email in EMAILS:
        print(f"  {email['id']} ...", end=" ", flush=True)
        row = ask(model, email)
        row["run"] = run_label
        row["match"] = row["answer"].lower().strip(".") == email["expect"]
        rows.append(row)
        print(f"{row['answer']:<10} {row['seconds']:6.2f}s")
    return rows


def run_model(model):
    payload = {"model": model, "runs": {}}
    for run_label in ("1", "2"):
        print(f"\n{model}  run {run_label}")
        payload["runs"][run_label] = run_pass(model, run_label)

    unload(model)
    path = RESULTS / (model.split(":")[0] + ".json")
    path.write_text(json.dumps(payload, indent=2))
    print(f"\nunloaded {model} -- saved {path}")
    return payload

In [ ]:
# Model 1. Nothing loaded, so run 1 is genuinely cold:  ollama stop qwen2.5:7b
QWEN = run_model("qwen2.5:7b")


qwen2.5:7b  run 1
  E01 ... 

In [ ]:
# Model 2. `ollama ps` should be empty before this runs.
LLAMA = run_model("llama3.1:8b")

## Reading the table

Nothing below re-runs a model — it reads the two JSON files. That is the point of writing them: a crash at email nine costs you one cell, not the afternoon.

**Run 1 versus run 2** — the gap is the model load; on a cold laptop it can be several times the warm time. Report run 2, mention run 1.

**`tok/s`** — the honest local speed metric. It is roughly your hardware, not the model, and barely moves between two 7B models on the same machine.

**`matched`** — the column that decides. A model that is 30% faster and 20% wrong is not faster, it is a queue of tickets for a human. Ten emails catches a clearly worse model; it cannot measure a 5% difference.

In [ ]:
def load(model):
    return json.loads((RESULTS / (model.split(":")[0] + ".json")).read_text())


def summarise(payload, run_label):
    rows = payload["runs"][run_label]
    times = sorted(r["seconds"] for r in rows)
    out_tokens = sum(r["tokens_out"] for r in rows)
    return {
        "median": times[len(times) // 2],
        "slowest": times[-1],
        "tok_per_s": out_tokens / sum(r["seconds"] for r in rows),
        "in": sum(r["tokens_in"] for r in rows) / len(rows),
        "out": out_tokens / len(rows),
        "matched": sum(r["match"] for r in rows),
    }


print(f"{'model':<14}{'run':>4}{'median':>8}{'slowest':>9}{'tok/s':>7}{'matched':>9}")
for name in MODELS:
    for run_label in ("1", "2"):
        s = summarise(load(name), run_label)
        print(f"{name:<14}{run_label:>4}{s['median']:>7.2f}s{s['slowest']:>8.2f}s"
              f"{s['tok_per_s']:>7.1f}{s['matched']:>6}/10")

## What this costs in the currency you actually spend

The `$/1,000 emails` figure is under a cent — and meaningless, because you are not billed. The real bill is **your time**: warm latency × the volume you would need.

In [ ]:
for volume in (1_000, 10_000):
    print(f"\n{volume:,} emails")
    for name in MODELS:
        median = summarise(load(name), "2")["median"]
        print(f"  {name:<14} {median * volume / 3600:6.1f} hours")

In [ ]:
def hosted_cost(name):
    s = summarise(load(name), "2")
    one = s["in"] / 1e6 * HOSTED_REFERENCE["price_in"] + s["out"] / 1e6 * HOSTED_REFERENCE["price_out"]
    return one * 1000


print(f"{'model':<14}{'$ now':>9}{'$/1,000 hosted':>18}")
for name in MODELS:
    print(f"{name:<14}{'0.00':>9}{hosted_cost(name):>18.4f}")

That pair of numbers is the honest cost of running locally: **zero naira and a day of wall-clock time per thousand emails.** Fine for ten, impossible for a production service.

And under a cent per thousand hosted either way — so **price is not the reason to run locally**. You run locally for privacy, offline work and zero sign-up; hosted for speed and quality. Keeping both numbers means that argument takes one sentence.

## The same model behind an API

The other half of "choosing a model **and API**" is the API shape, and Ollama also serves an **OpenAI-compatible** endpoint: `http://127.0.0.1:11434/v1/chat/completions`. Same model, different route, different JSON — and it is the shape nearly every hosted provider speaks.

In [ ]:
def ask_v1(model, email):
    response = httpx2.post(
        OLLAMA_V1,
        json={
            "model": model,
            "messages": [
                {"role": "system", "content": SYSTEM},
                {"role": "user", "content": build_prompt(email)},
            ],
            "temperature": 0,
        },
        timeout=300,
    )
    data = response.json()
    return {
        "answer": data["choices"][0]["message"]["content"].strip(),
        "tokens_in": data["usage"]["prompt_tokens"],
        "tokens_out": data["usage"]["completion_tokens"],
    }


print(ask_v1("qwen2.5:7b", EMAILS[0]))

| | `/api/chat` (Ollama) | `/v1/chat/completions` (OpenAI-shaped) |
|---|---|---|
| the answer | `message.content` | `choices[0].message.content` |
| input tokens | `prompt_eval_count` | `usage.prompt_tokens` |
| output tokens | `eval_count` | `usage.completion_tokens` |
| settings | `options: {temperature, num_ctx}` | flat `temperature`, no `num_ctx` |
| auth | none, local | `Authorization: Bearer <key>` |

Moving to a hosted provider is therefore: change the base URL, add a key header, rename two fields. That is the whole migration — and why you write one `ask()` and not one per provider.

## Ask for JSON, then validate it

Free text is hard to use. Ollama can constrain the reply with `"format": "json"` — but valid JSON is not *correct* JSON. Two failures, both of which count:

- **not JSON** — the reply came back as prose. `format: "json"` makes this rare, not impossible.
- **JSON but wrong** — `{"cat": "urgent"}` when you asked for `category`. Parsing succeeds; your code breaks later.

In [ ]:
PROMPT_JSON = ('Return JSON only, in the shape '
               '{"category": "urgent|billing|access|general|spam"}. ')

labels = {"urgent", "billing", "access", "general", "spam"}
parsed, broken = 0, 0

response = httpx2.post(
    OLLAMA_CHAT,
    json={
        "model": "qwen2.5:7b",
        "messages": [{"role": "user", "content": PROMPT_JSON + build_prompt(EMAILS[0])}],
        "stream": False,
        "format": "json",
        "options": {"temperature": 0, "num_ctx": 8192},
    },
    timeout=300,
)
try:
    body = json.loads(response.json()["message"]["content"])
    parsed += 1
    if body.get("category") not in labels:
        broken += 1
        print("parsed but invalid:", body)
except json.JSONDecodeError:
    broken += 1
    print("not JSON at all")

print("parsed:", parsed, "| unusable:", broken)
unload("qwen2.5:7b")

Validating what comes back is next week's problem: schema, reject, retry. The habit to start now is **counting failures instead of assuming there are none**.

## The decision, and the weaker laptop

Three lines close this exercise:

```
Chose: qwen2.5:7b
Because: 9/10 matched vs 8/10, warm median 6.2s vs 7.4s, both fit in 6 GB
Caveat: 10 emails, one CPU-only laptop, 2 of 10 in Pidgin, q4_K_M build
```

That is a decision. "It felt better" is not, and neither is a screenshot of a benchmark from the internet. The caveat tells the next person when to re-run this — Week 14 for this team.

On 8 GB, use a 4B model and say so in your note; a documented 4B result beats a comparison you could not finish. To share one machine, set `OLLAMA_HOST=0.0.0.0:11434` there and call `http://<that-ip>:11434/api/chat` — **but Ollama has no login**, so private network only, and set it back afterwards.

| Symptom | Cause | Fix |
|---|---|---|
| `connection refused` on 11434 | server not running | open the app, or `ollama serve` |
| `model not found` | not pulled, or tag wrong | `ollama pull qwen2.5:7b` |
| replies cut off | `num_ctx` too small | raise it, or shorten the prompt |
| laptop freezes | two models loaded | `OLLAMA_MAX_LOADED_MODELS=1`, close Docker |
| very slow | CPU-only, normal | `ollama ps` showing `100% CPU` is expected |

## Summary

| Idea | Rule |
|---|---|
| Fair comparison | same emails, same prompt, `temperature: 0`, same `num_ctx` |
| Expected answers | written before the run |
| Tokens | `prompt_eval_count` and `eval_count`; `total_duration` is nanoseconds |
| Two runs | run 1 pays the load, run 2 is the number you quote |
| Speed | `tok/s`, because it travels between machines |
| Cost | zero in money, hours of your time — plus a hosted price to compare |
| Limits | memory, context, one model at a time; no 429, just a queue |
| API | `/api/chat` and `/v1/chat/completions` differ in fields, not in idea |
| Decision | model, numbers, caveat — re-run at Week 14 |

One line: **a local model costs no money and all of your patience — measure both, then choose.**